# One way ANOVA

In [0]:
import pandas as pd
# import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
np.random.seed(42)

In [0]:
import numpy as np
import pandas as pd
from scipy import stats

def anova(*args):
    # Number of groups and total observations
    k = len(args)
    N = sum(len(arg) for arg in args)
    
    # Means of each group
    group_means = [np.mean(arg) for arg in args]
    overall_mean = np.mean([x for group in args for x in group])
    
    # Between-group sum of squares (SSB)
    ssb = sum(len(arg) * (np.mean(arg) - overall_mean) ** 2 for arg in args)
    
    # Within-group sum of squares (SSW)
    ssw = sum(sum((x - np.mean(arg)) ** 2 for x in arg) for arg in args)
    
    # Total sum of squares (SST)
    sst = ssb + ssw
    
    # Degrees of freedom
    df_between = k - 1
    df_within = N - k
    
    # Mean squares
    msb = ssb / df_between
    mse = ssw / df_within
    
    # F-statistic and p-value
    F = msb / mse
    p_value = stats.f.sf(F, df_between, df_within)
    
    # Print means
    print("Group Means:")
    for i, m in enumerate(group_means, 1):
        print(f"  Mean of Group {i}: {m:.2f}")
    print(f"\nOverall Mean: {overall_mean:.2f}\n")
    
    # ANOVA summary table
    table = pd.DataFrame({
        "Source": ["Between Groups", "Within Groups", "Total"],
        "SS": [ssb, ssw, sst],
        "df": [df_between, df_within, df_between + df_within],
        "MS": [msb, mse, None],
        "F": [F, None, None],
        "p-value": [p_value, None, None]
    })

    F_upper = stats.f.ppf(0.95, df_between, df_within)
    print(f"F Upper: {F_upper:.2f}")
    if F > F_upper:
        print("Reject the null hypothesis. There is a significant difference between the group means.")
    else:
        print("Fail to reject the null hypothesis. There is no significant difference between the group means.")
    
    return table

# Example Data
result = anova(
    [254, 263, 241, 237, 251],
    [234, 218, 235, 227, 216],
    [200, 222, 197, 206, 204]
)

print(result.round(4))


# Two Way ANOVA

In [0]:
import numpy as np
import pandas as pd
from scipy import stats

def two_way_anova(data):
    """
    data: 2D list or numpy array
    rows = levels of Factor A
    columns = levels of Factor B
    each cell contains multiple observations
    shape = (a, b, n)
    """

    data = np.array(data)
    
    a, b, n = data.shape  # levels of A, B, observations
    
    grand_mean = np.mean(data)
    
    # Means
    mean_A = np.mean(data, axis=(1,2))
    mean_B = np.mean(data, axis=(0,2))
    mean_cell = np.mean(data, axis=2)

    # Sum of Squares
    SSA = b*n * np.sum((mean_A - grand_mean)**2)
    SSB = a*n * np.sum((mean_B - grand_mean)**2)

    SSAB = n * np.sum((mean_cell 
                       - mean_A[:,None] 
                       - mean_B[None,:] 
                       + grand_mean)**2)

    SSE = np.sum((data - mean_cell[:,:,None])**2)

    SST = SSA + SSB + SSAB + SSE

    # Degrees of freedom
    df_A = a - 1
    df_B = b - 1
    df_AB = (a - 1)*(b - 1)
    df_E = a*b*(n - 1)

    # Mean Squares
    MSA = SSA / df_A
    MSB = SSB / df_B
    MSAB = SSAB / df_AB
    MSE = SSE / df_E

    # F statistics
    FA = MSA / MSE
    FB = MSB / MSE
    FAB = MSAB / MSE

    # p-values
    pA = stats.f.sf(FA, df_A, df_E)
    pB = stats.f.sf(FB, df_B, df_E)
    pAB = stats.f.sf(FAB, df_AB, df_E)

    table = pd.DataFrame({
        "Source": ["Factor A", "Factor B", "Interaction", "Error", "Total"],
        "SS": [SSA, SSB, SSAB, SSE, SST],
        "df": [df_A, df_B, df_AB, df_E, df_A+df_B+df_AB+df_E],
        "MS": [MSA, MSB, MSAB, MSE, None],
        "F": [FA, FB, FAB, None, None],
        "p-value": [pA, pB, pAB, None, None]
    })

    return table


# Example Data
# Factor A = Teaching Method (2 levels)
# Factor B = Study Hours (3 levels)
# 3 observations per cell

data = [
    [[85,88,90], [78,80,82], [92,94,96]],   # Method 1
    [[83,85,87], [76,79,81], [89,91,93]]    # Method 2
]

result = two_way_anova(data)
print(result.round(4))

**Tukey Cramer Critical Range**

# Variance, Covariance

# Correlation, Auto correlation